# Explain a distance through its matching

Suppose two computations give barcodes whose bottleneck distance is one. We want to know which intervals account for that answer. The bottleneck distance is the smallest possible largest cost of pairing their intervals, allowing a finite interval to be paired to the diagonal at half its length. We will keep an assignment attaining the distance, draw its costs, and compare them with a calculation we can do by hand.

Use a Julia environment with TamerOp and CairoMakie; the [installation page](../src/start/install.md) explains setup. The intervals below use the same units in an increasing parameter. When starting from ordinary persistence diagrams, choose the same homological degree and compatible parameter conventions, as in the [ordinary persistence guide](../ordinary_persistence.md).


In [ ]:
using TamerOp, CairoMakie
const OP = TamerOp
const OA = TamerOp.Advanced


## Retain the assignment

Barcode A has two copies of `[0,2)`, the interval `[8,10)`, and an interval born at four that never dies. Barcode B has two copies of `[0,3)` and an interval born at five that never dies. We call intervals with infinite death *essential*. Equal copies retain their multiplicity: two copies describe two independent classes even when their points coincide in the diagram.

The `Advanced` call returns a matching witness, which retains the chosen partners and their costs as well as the distance.


In [ ]:
a = [(0.0, 2.0), (0.0, 2.0), (8.0, 10.0), (4.0, Inf)]
b = [(0.0, 3.0), (0.0, 3.0), (5.0, Inf)]
witness = OA.bottleneck_matching(a, b)
@assert witness.distance == 1
witness.distance


Pair the two `[0,2)` copies with the two `[0,3)` copies. Each costs one, the largest absolute difference between its birth and death endpoints. The unmatched `[8,10)` costs half its length, again one. Finally, pairing the essential intervals costs the difference between their births, also one. These choices show that distance one is attainable. Every finite-cost matching must pair the two essential intervals, so their birth difference prevents any smaller answer.

The returned assignment attains this optimum, although exchanging equal copies gives another equally good assignment. Their identical endpoints provide no further way to identify individual classes.


In [ ]:
OP.visualize(witness)


The dashed connector sends `[8,10)` to the birth–death diagonal. In the barcode panel its empty partner row records the same choice. Essential intervals appear on a labelled infinity rail so they can be drawn in a finite figure; their death endpoints remain infinite in the matching.


In [ ]:
late_pair = findfirst(==((8.0, 10.0)), witness.points_a)
@assert witness.a_to_b[late_pair] == 0
OP.visualize(witness; pair=late_pair)


## Remove one essential interval

Now remove the essential interval from B. The remaining essential interval in A cannot be paired with a finite interval or the diagonal at finite cost. The distance should therefore become infinite. Keeping the original inputs lets us compare this change directly.


In [ ]:
finite_b = b[1:2]
mismatch = OA.bottleneck_matching(a, finite_b)
@assert isinf(mismatch.distance)
mismatch.distance


The infinite answer records the mismatch between essential intervals. Removing them from both inputs would define a different comparison. For ordinary diagrams, `inspection_session(diagram_a, diagram_b; dim=1)` selects first homology and applies its declared essential-interval policy to that comparison.

## Optional: inspect the members interactively

With WGLMakie loaded and Julia running, linked charts let you select a point or barcode row and inspect its partner. Repeated clicks on coincident points cycle through their separate members, while the pair-ID field reaches members omitted by a drawing limit. Hovering reads already retained costs; it does not recompute persistence or find a new matching.


```julia
using WGLMakie
session = OP.inspection_session(witness)
OP.visualize(session; backend=:wglmakie)
```

Use `OP.visualize(OA.inspection_snapshot(session); backend=:cairomakie)` for a static figure of the selected view. Once you finish, `OA.close_inspection!(session)` releases the live inputs and callbacks; the final snapshot remains available.

For two finite encodings, the [finite window matching guide](../exact_matching.md#retain-and-inspect-the-witness) follows the same comparison across slices and retains a witness for the exact optimum. A pairing of intervals compares these summaries. A correspondence between source cycles, or an interleaving between the modules themselves, needs additional information.
